In [55]:

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import argparse
import json
import os
import subprocess
import sys
from datetime import date, datetime, timedelta
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import ssl
import certifi

In [56]:
# ── Config ────────────────────────────────────────────────────────────────────

def get_ridb_api_key():
    user = os.environ.get("USER") or os.environ.get("LOGNAME") or ""
    if not user:
        return None
    try:
        return subprocess.check_output(
            [
                "security",
                "find-generic-password",
                "-a",
                user,
                "-s",
                "recreation-gov-api",
                "-w",
            ],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

RIDB_API_KEY = get_ridb_api_key() or os.environ.get("RIDB_API_KEY")
if not RIDB_API_KEY:
    raise RuntimeError(
        "RIDB API key not found in macOS keychain 'recreation-gov-api'. "
        "Add it with: security add-generic-password -a \"$USER\" -s \"recreation-gov-api\" -w \"<API_KEY>\""
    )

RIDB_BASE    = "https://ridb.recreation.gov/api/v1"
REC_BASE     = "https://www.recreation.gov/api"
WEATHER_BASE = "https://api.open-meteo.com/v1"
USER_AGENT   = "CampsiteScout/1.0 (dev)"

# Point Reyes — hardcoded for POC; add more in facility-ids.md
POINT_REYES = {
    "name": "Point Reyes National Seashore",
    "lat": 38.037,
    "lon": -122.803,
    "camps": [
        {"name": "Sky Camp",     "facility_id": "234059", "permit_id": "4675310"},
        {"name": "Coast Camp",   "facility_id": "234061", "permit_id": "4675311"},
        {"name": "Glen Camp",    "facility_id": "234060", "permit_id": "4675312"},
        {"name": "Wildcat Camp", "facility_id": "234062", "permit_id": "4675313"},
    ],
}

WMO_LABELS = {
    0: "Clear ☀️", 1: "Mainly clear 🌤️", 2: "Partly cloudy ⛅", 3: "Overcast ☁️",
    45: "Fog 🌫️", 48: "Icy fog 🌫️",
    51: "Light drizzle 🌦️", 53: "Drizzle 🌦️", 55: "Heavy drizzle 🌦️",
    61: "Light rain 🌧️", 63: "Rain 🌧️", 65: "Heavy rain 🌧️",
    71: "Light snow ❄️", 73: "Snow ❄️", 75: "Heavy snow ❄️", 77: "Snow grains 🌨️",
    80: "Showers 🌦️", 81: "Showers 🌦️", 82: "Heavy showers 🌦️",
    85: "Snow showers 🌨️", 86: "Heavy snow showers 🌨️",
    95: "Thunderstorm ⛈️", 96: "Thunderstorm + hail ⛈️", 99: "Thunderstorm + hail ⛈️",
}

In [57]:
# ── Helpers ─────────────────────────────────────────────────────────────────

def divider(title="", width=70, char="─"):
    if title:
        pad = (width - len(title) - 2) // 2
        print(f"{char * pad} {title} {char * (width - pad - len(title) - 2)}")
    else:
        print(char * width)

def fetch(url, headers=None, label=None):
    """Fetch URL, return (parsed_json, raw_text). Prints the request URL."""
    req = Request(url, headers={
        "User-Agent": USER_AGENT,
        "Accept": "application/json",
        **(headers or {}),
    })
    if label:
        print(f"\n🌐 {label}")
    print(f"   GET {url}")
    try:
        context = ssl.create_default_context(cafile=certifi.where())
        with urlopen(req, context=context, timeout=15) as resp:
            raw = resp.read().decode("utf-8")
            data = json.loads(raw)
            print(f"   ✅ {resp.status} — {len(raw):,} bytes")
            return data, raw
    except HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")[:300]
        print(f"   ❌ HTTP {e.code}: {body}")
        return None, None
    except URLError as e:
        print(f"   ❌ Network error: {e.reason}")
        return None, None

def pprint(data, indent=2):
    print(json.dumps(data, indent=indent, default=str))

def first_of_month(year_month: str) -> str:
    """'2026-04' → '2026-04-01T00:00:00.000Z'"""
    return f"{year_month}-01T00:00:00.000Z"

def this_weekend():
    """Return (friday, sunday) for the upcoming weekend."""
    today = date.today()
    days_until_friday = (4 - today.weekday()) % 7
    if days_until_friday == 0:
        days_until_friday = 7
    friday = today + timedelta(days=days_until_friday)
    sunday = friday + timedelta(days=2)
    return friday, sunday

def parse_weekend_avail(availabilities: dict, friday: date, sunday: date) -> dict:
    """Extract Fri/Sat/Sun availability from a campsites dict."""
    target_dates = {friday, friday + timedelta(1), sunday}
    result = {}
    for dt_str, status in availabilities.items():
        try:
            dt = datetime.fromisoformat(dt_str.replace("Z", "+00:00")).date()
        except Exception:
            continue
        if dt in target_dates:
            result[dt.isoformat()] = status
    return result

In [58]:
# ── API calls ─────────────────────────────────────────────────────────────────

def search_ridb_facilities(query: str, show_raw=False):
    divider(f"RIDB SEARCH — '{query}'")
    url = (
        f"{RIDB_BASE}/facilities"
        f"?query={query.replace(' ', '+')}"
        f"&facilitytype=Campground&radius=30&limit=10"
        f"&apikey={RIDB_API_KEY}"
    )
    data, _ = fetch(url, label="Facility search")
    if not data:
        return []

    if show_raw:
        print("\n── Raw JSON ──")
        pprint(data)

    results = data.get("RECDATA", [])
    print(f"\n{'ID':<12} {'URL':<60}  Name")
    print("─" * 110)
    for f in results:
        fid  = f.get("FacilityID", "?")
        name = f.get("FacilityName", "?")
        campground_url = f"https://www.recreation.gov/camping/campgrounds/{fid}"
        print(f"{fid:<12} {campground_url:<60}  {name}")
    return results

def check_campground_availability(facility_id: str, year_month: str, friday: date, sunday: date, show_raw=False):
    """Try the standard campground endpoint."""
    divider(f"AVAILABILITY — Facility {facility_id} ({year_month})")
    url = (
        f"{REC_BASE}/camps/availability/campground/{facility_id}/month"
        f"?start_date={first_of_month(year_month)}"
    )
    data, _ = fetch(url, label="Campground availability")
    if not data:
        return None

    if show_raw:
        print("\n── Raw JSON (first 2 campsites) ──")
        campsites = data.get("campsites", {})
        sample = dict(list(campsites.items())[:2])
        pprint({"campsites": sample})

    # Summarize weekend availability
    campsites = data.get("campsites", {})
    summary = {}   # date → {"available": N, "total": N}
    for site_id, site in campsites.items():
        avails = site.get("availabilities", {})
        weekend = parse_weekend_avail(avails, friday, sunday)
        for dt, status in weekend.items():
            summary.setdefault(dt, {"available": 0, "total": 0})
            summary[dt]["total"] += 1
            if status in ("Available", "Open"):
                summary[dt]["available"] += 1

    if summary:
        print(f"\n{'Date':<14} {'Open':>6} {'Total':>7}  Status")
        print("─" * 40)
        for dt in sorted(summary):
            a = summary[dt]["available"]
            t = summary[dt]["total"]
            icon = "✅" if a > 0 else "❌"
            print(f"{dt:<14} {a:>6} {t:>7}  {icon}")
    else:
        print("\n  No data for target dates.")
    return data

def check_permit_availability(permit_id: str, year_month: str, friday: date, sunday: date, show_raw=False):
    """Try the wilderness permit endpoint."""
    divider(f"PERMIT AVAILABILITY — Permit {permit_id} ({year_month})")
    url = (
        f"{REC_BASE}/permits/{permit_id}/availability/month"
        f"?start_date={first_of_month(year_month)}&commercial_acct=false"
    )
    data, _ = fetch(url, label="Permit availability")
    if not data:
        return None

    if show_raw:
        print("\n── Raw JSON ──")
        pprint(data)

    # Permit responses vary — try to find availability payload
    payload = data.get("payload", data)
    avail   = payload.get("availability", {})

    if not avail:
        print("\n  Unexpected response shape — printing top-level keys:")
        print(" ", list(data.keys()))
        return data

    print(f"\n{'Date':<14} {'Remaining':>10} {'Total':>7}  Status")
    print("─" * 45)
    target_dates = {friday, friday + timedelta(1), sunday}
    for dt_str, info in sorted(avail.items()):
        try:
            dt = datetime.fromisoformat(dt_str.replace("Z", "+00:00")).date()
        except Exception:
            continue
        if dt not in target_dates:
            continue
        remaining = info.get("remaining", "?")
        total     = info.get("total", "?")
        status    = info.get("status", "?")
        icon      = "✅" if isinstance(remaining, int) and remaining > 0 else "❌"
        print(f"{dt.isoformat():<14} {str(remaining):>10} {str(total):>7}  {icon} {status}")

    return data

def get_weather(lat: float, lon: float, friday: date, sunday: date, show_raw=False):
    divider(f"WEATHER — ({lat}, {lon})")
    url = (
        f"{WEATHER_BASE}/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&daily=weathercode,temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max"
        f"&temperature_unit=fahrenheit&wind_speed_unit=mph&precipitation_unit=inch"
        f"&timezone=auto&forecast_days=14"
    )
    data, _ = fetch(url, label="Open-Meteo forecast")
    if not data:
        return None

    if show_raw:
        print("\n── Raw JSON ──")
        pprint(data)

    daily   = data.get("daily", {})
    times   = daily.get("time", [])
    codes   = daily.get("weathercode", [])
    t_max   = daily.get("temperature_2m_max", [])
    t_min   = daily.get("temperature_2m_min", [])
    precip  = daily.get("precipitation_sum", [])
    wind    = daily.get("windspeed_10m_max", [])

    target_dates = {
        friday.isoformat(),
        (friday + timedelta(1)).isoformat(),
        sunday.isoformat(),
    }

    print(f"\n{'Date':<14} {'High':>6} {'Low':>6} {'Precip':>8} {'Wind':>8}  Condition")
    print("─" * 72)
    for i, t in enumerate(times):
        if t not in target_dates:
            continue
        code  = codes[i] if i < len(codes) else None
        label = WMO_LABELS.get(code, f"Code {code}")
        hi    = f"{t_max[i]:.0f}°F" if i < len(t_max) else "—"
        lo    = f"{t_min[i]:.0f}°F" if i < len(t_min) else "—"
        pr    = f"{precip[i]:.2f}\"" if i < len(precip) and precip[i] else "0\""
        wi    = f"{wind[i]:.0f} mph" if i < len(wind) else "—"
        warn  = " ⚠️ HIGH WIND" if i < len(wind) and wind[i] > 25 else ""
        print(f"{t:<14} {hi:>6} {lo:>6} {pr:>8} {wi:>8}  {label}{warn}")

    return data

In [ ]:
# Interactive Widgets

area_input = widgets.Text(description='Area:', placeholder='e.g., big sur')
facility_input = widgets.Text(description='Facility ID:', placeholder='e.g., 234059')
permit_input = widgets.Text(description='Permit ID:', placeholder='e.g., 4675310')
month_input = widgets.Text(description='Month:', value=date.today().strftime('%Y-%m'), placeholder='e.g., 2026-06')
raw_checkbox = widgets.Checkbox(description='Show Raw JSON')

run_button = widgets.Button(description='Run Check')
output_area = widgets.Output()

def on_run_button_clicked(b):
    with output_area:
        clear_output()
        
        friday, sunday = this_weekend()
        year_month = month_input.value or date.today().strftime('%Y-%m')
        
        print('=' * 70)
        print('  🏕️  Campsite API Checker')
        print(f'  Weekend: {friday} (Fri) → {sunday} (Sun)')
        print(f'  Month:   {year_month}')
        print('=' * 70)
        
        if area_input.value:
            area = area_input.value.strip().lower()
            if area == "point reyes":
                # Use hardcoded Point Reyes camps
                lat, lon = POINT_REYES['lat'], POINT_REYES['lon']
                
                for camp in POINT_REYES['camps']:
                    print(f"\n{'━' * 70}")
                    print(f"  {camp['name']}")
                    print(f"{'━' * 70}")
                    
                    result = check_campground_availability(
                        camp['facility_id'], year_month, friday, sunday, show_raw=raw_checkbox.value
                    )
                    if result is None or not result.get('campsites'):
                        print("\n  → Standard endpoint empty/failed, trying permit endpoint...")
                        check_permit_availability(
                            camp['permit_id'], year_month, friday, sunday, show_raw=raw_checkbox.value
                        )
                
                get_weather(lat, lon, friday, sunday, show_raw=raw_checkbox.value)
                
                divider('DONE')
                print('\n  Tip: Check "Show Raw JSON" to dump full JSON.')
                print('  Tip: Enter an area to discover facility IDs.')
                return
            else:
                # Search RIDB for other areas
                search_ridb_facilities(area_input.value, show_raw=raw_checkbox.value)
                return
        
        if facility_input.value or permit_input.value:
            if facility_input.value:
                check_campground_availability(facility_input.value, year_month, friday, sunday, show_raw=raw_checkbox.value)
            if permit_input.value:
                check_permit_availability(permit_input.value, year_month, friday, sunday, show_raw=raw_checkbox.value)
            return
        
        # Default: Point Reyes
        lat, lon = POINT_REYES['lat'], POINT_REYES['lon']
        
        for camp in POINT_REYES['camps']:
            print(f"\n{'━' * 70}")
            print(f"  {camp['name']}")
            print(f"{'━' * 70}")
            
            result = check_campground_availability(
                camp['facility_id'], year_month, friday, sunday, show_raw=raw_checkbox.value
            )
            if result is None or not result.get('campsites'):
                print("\n  → Standard endpoint empty/failed, trying permit endpoint...")
                check_permit_availability(
                    camp['permit_id'], year_month, friday, sunday, show_raw=raw_checkbox.value
                )
        
        get_weather(lat, lon, friday, sunday, show_raw=raw_checkbox.value)
        
        divider('DONE')
        print('\n  Tip: Check "Show Raw JSON" to dump full JSON.')
        print('  Tip: Enter an area to discover facility IDs.')

run_button.on_click(on_run_button_clicked)

display(area_input, facility_input, permit_input, month_input, raw_checkbox, run_button, output_area)

Text(value='', description='Area:', placeholder='e.g., big sur')

Text(value='', description='Facility ID:', placeholder='e.g., 234059')

Text(value='', description='Permit ID:', placeholder='e.g., 4675310')

Text(value='2026-04', description='Month:', placeholder='e.g., 2026-06')

Checkbox(value=False, description='Show Raw JSON')

Button(description='Run Check', style=ButtonStyle())

Output()